In [1]:
# @title Setup: local library, Google Drive, or GitHub
import os
import subprocess
import sys
from pathlib import Path

USE_GOOGLE_DRIVE = False  # @param {type:"boolean"}
USE_GITHUB = False  # @param {type:"boolean"}
REPO_URL = "https://github.com/Baoshan-Song/KFV-FGO-Comparison.git"
BRANCH = "python_colab"
PROJECT_NAME = "KFV-FGO-Comparison"

DRIVE_ROOT = Path("/content/drive/MyDrive")
if USE_GOOGLE_DRIVE:
  try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Google Drive mounted:", DRIVE_ROOT.exists())
  except ImportError:
    print("No Colab runtime; Google Drive mount skipped.")

if USE_GITHUB:
  root = Path("/content") if Path("/content").exists() else Path.cwd().parent
  PROJECT_PATH = root / PROJECT_NAME
  if not PROJECT_PATH.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL,
                    str(PROJECT_PATH)], check=True)
else:
  candidates = [
      DRIVE_ROOT / "Colab Notebooks" / "KFV-FGO-Comparison-main",
      Path.cwd(),
      Path.home() / "Library/CloudStorage/GoogleDrive-zsyqdl@gmail.com/我的云端硬盘/Colab Notebooks/KFV-FGO-Comparison-main",
  ]
  PROJECT_PATH = next(
      (path for path in candidates if (path / "experiment_common.py").exists()),
      None,
  )
  if PROJECT_PATH is None:
    raise FileNotFoundError("Cannot locate the local KFV-FGO project")

os.chdir(PROJECT_PATH)
if str(PROJECT_PATH) not in sys.path:
  sys.path.insert(0, str(PROJECT_PATH))
print(f"Using project: {PROJECT_PATH}")

Using project: /Users/tommy/Library/CloudStorage/GoogleDrive-zsyqdl@gmail.com/我的云端硬盘/Colab Notebooks/KFV-FGO-Comparison-main


In [2]:
# @title 1. KFV vs KFV
# @markdown Simulation parameters and both KFV estimator parameters are adjustable. Results play side-by-side below.
from IPython.display import display
import importlib
import experiment_common
importlib.reload(experiment_common)
from experiment_common import animate_pair, make_config, make_data, run_and_display

# --------------------------------------------------
# Simulation Data Parameters
# --------------------------------------------------
steps = 100  # @param {type:"slider", min:30, max:160, step:10}  # Unit: steps
distance = 105  # @param {type:"slider", min:200, max:1000, step:50}  # Unit: m
seed = 7  # @param {type:"integer", min:0, max:999}
outlier_weight = 0.  # @param {type:"slider", min:0, max:1, step:0.05}  # Ratio (0-1)
outlier_mean = 0.0  # @param {type:"number"}  # Unit: m
outlier_sigma = 10.0  # @param {type:"number"}  # Unit: m
white_sigma = 0.1  # @param {type:"number"}  # Unit: m

# --------------------------------------------------
# Left Column KFV Configuration
# --------------------------------------------------
left_mode = "EKF"  # @param ["EKF", "iEKF", "rEKF", "riEKF"]
left_iterations = 2  # @param {type:"integer", min:1, max:20}
left_kernel = "huber"  # @param ["none", "huber", "cauchy", "tukey"]
left_delta = 2.0  # @param {type:"number"}

# --------------------------------------------------
# Right Column KFV Configuration
# --------------------------------------------------
right_mode = "riEKF"  # @param ["EKF", "iEKF", "rEKF", "riEKF"]
right_iterations = 20  # @param {type:"integer", min:1, max:20}
right_kernel = "huber"  # @param ["none", "huber", "cauchy", "tukey"]
right_delta = 2.0  # @param {type:"number"}

# --------------------------------------------------
# Helper Functions and Execution
# --------------------------------------------------
def make_ui_config(mode, iterations, kernel, delta):
  return make_config(kfv_mode=mode, max_iteration=iterations, robust_kernel=kernel, robust_delta=delta)

sim_data = make_data(seed=seed, num_steps=steps, anchor_radius=distance,
                     outlier_weight=outlier_weight, outlier_mean=outlier_mean,
                     outlier_sigma=outlier_sigma, white_sigma=white_sigma)

left = {"name": f"KFV-{left_mode}", "kind": "kfv", "config": make_ui_config(
    left_mode, left_iterations, left_kernel, left_delta)}

right = {"name": f"KFV-{right_mode}", "kind": "kfv", "config": make_ui_config(
    right_mode, right_iterations, right_kernel, right_delta)}

experiment = run_and_display(sim_data, left, right)
display(animate_pair(experiment, interval=80))


🚀 [Run Estimator 1]: KFV-EKF (kfv)
🚀 [Run Estimator 2]: KFV-riEKF (kfv)
  [DEBUG Metrics] 估计轨迹 Shape: (4, 100) | 真值 Shape: (2, 100) | 对齐计算帧数: 100
  [DEBUG Metrics] 估计轨迹 Shape: (4, 100) | 真值 Shape: (2, 100) | 对齐计算帧数: 100

统计结果与维度诊断列表:
KFV-EKF: RMSE=15.269 m | MAE=2.692 m | CP95=8.265 m | Max=141.421 m | Time=205.50 ms (2.05 ms/step)
KFV-riEKF: RMSE=14.142 m | MAE=1.482 m | CP95=0.136 m | Max=141.421 m | Time=2544.91 ms (25.45 ms/step)



In [ ]:
# @title 2. KFV vs FGO
# @markdown Simulation parameters, KFV parameters, and FGO parameters are all adjustable. Results play side-by-side below.
from IPython.display import display
import importlib
import experiment_common
# importlib.reload(experiment_common)
from experiment_common import animate_pair, make_config, make_data, run_and_display

# --------------------------------------------------
# Simulation Data Parameters
# --------------------------------------------------
steps = 100  # @param {type:"slider", min:30, max:160, step:10}  # Unit: steps
distance = 105  # @param {type:"slider", min:200, max:1000, step:50}  # Unit: m
seed = 7  # @param {type:"integer", min:0, max:999}
outlier_weight = 0.35  # @param {type:"slider", min:0, max:1, step:0.05}  # Ratio (0-1)
outlier_mean = 30.0  # @param {type:"number"}  # Unit: m
outlier_sigma = 5.0  # @param {type:"number"}  # Unit: m
white_sigma = 0.1  # @param {type:"number"}  # Unit: m

# --------------------------------------------------
# Left Column KFV Configuration
# --------------------------------------------------
kfv_mode = "EKF"  # @param ["EKF", "iEKF", "rEKF", "riEKF"]
kfv_iterations = 1  # @param {type:"integer", min:1, max:20}
kfv_kernel = "none"  # @param ["none", "huber", "cauchy", "tukey"]
kfv_delta = 2.0  # @param {type:"number"}

# --------------------------------------------------
# Right Column FGO Configuration
# --------------------------------------------------
fgo_iterations = 1  # @param {type:"integer", min:1, max:20}
fgo_kernel = "huber"  # @param ["none", "huber", "cauchy", "tukey"]
fgo_delta = 2.0  # @param {type:"number"}
fgo_imitate_kfv = True  # @param {type:"boolean"}
fgo_window_size = 1  # @param {type:"integer", min:1, max:20}
fgo_auto_diff = False  # @param {type:"boolean"}

# --------------------------------------------------
# Execution with Default Configurations
# --------------------------------------------------
sim_data = make_data(seed=seed, num_steps=steps, anchor_radius=distance,
                     outlier_weight=outlier_weight, outlier_mean=outlier_mean,
                     outlier_sigma=outlier_sigma, white_sigma=white_sigma)

# 完全依赖 make_config() 的默认参数创建 Config 实例
left_config = make_config(
)

right_config = make_config(
    imitate_kfv=fgo_imitate_kfv,
)

left = {"name": f"KFV-{kfv_mode}", "kind": "kfv", "config": left_config}
right = {"name": "Re-FGO", "kind": "fgo", "config": right_config}

experiment = run_and_display(sim_data, left, right)
display(animate_pair(experiment, interval=80))


# # @title 2. KFV vs FGO
# # @markdown Simulation parameters, KFV parameters, and FGO parameters are all adjustable. Results play side-by-side below.
# from IPython.display import display
# import importlib
# import experiment_common
# # importlib.reload(experiment_common)
# from experiment_common import animate_pair, make_config, make_data, run_and_display

# # --------------------------------------------------
# # Simulation Data Parameters
# # --------------------------------------------------
# steps = 100  # @param {type:"slider", min:30, max:160, step:10}  # Unit: steps
# distance = 105  # @param {type:"slider", min:200, max:1000, step:50}  # Unit: m
# seed = 7  # @param {type:"integer", min:0, max:999}
# outlier_weight = 0.35  # @param {type:"slider", min:0, max:1, step:0.05}  # Ratio (0-1)
# outlier_mean = 30.0  # @param {type:"number"}  # Unit: m
# outlier_sigma = 5.0  # @param {type:"number"}  # Unit: m
# white_sigma = 0.1  # @param {type:"number"}  # Unit: m

# # --------------------------------------------------
# # Left Column KFV Configuration
# # --------------------------------------------------
# kfv_mode = "EKF"  # @param ["EKF", "iEKF", "rEKF", "riEKF"]
# kfv_iterations = 1  # @param {type:"integer", min:1, max:20}
# kfv_kernel = "none"  # @param ["none", "huber", "cauchy", "tukey"]
# kfv_delta = 2.0  # @param {type:"number"}

# # --------------------------------------------------
# # Right Column FGO Configuration
# # --------------------------------------------------
# fgo_iterations = 1  # @param {type:"integer", min:1, max:20}
# fgo_kernel = "huber"  # @param ["none", "huber", "cauchy", "tukey"]
# fgo_delta = 2.0  # @param {type:"number"}
# fgo_imitate_kfv = True  # @param {type:"boolean"}
# fgo_window_size = 1  # @param {type:"integer", min:1, max:20}
# fgo_auto_diff = False  # @param {type:"boolean"}

# # --------------------------------------------------
# # Helper Functions and Execution
# # --------------------------------------------------
# def make_ui_config(mode, iterations, kernel, delta, imitate=False, window=1, autodiff=False):
#   return make_config(
#       err_x=100.0, err_y=-100.0, err_vx=0.0, err_vy=0.0,
#       p0_diag=(50.0, 50.0, 1.0, 1.0),
#       kfv_mode=mode,  # 当 imitate_kfv 为 True 时，需要与 KFV 模式 (如 riEKF) 匹配
#       robust_kernel=kernel,
#       robust_delta=delta,
#       max_iteration=iterations,
#       window_size=window,
#       imitate_kfv=imitate,
#       autodiff=autodiff
#   )

# sim_data = make_data(seed=seed, num_steps=steps, anchor_radius=distance,
#                      outlier_weight=outlier_weight, outlier_mean=outlier_mean,
#                      outlier_sigma=outlier_sigma, white_sigma=white_sigma)

# # 左侧估计器配置 (KFV)
# left = {"name": f"KFV-{kfv_mode}", "kind": "kfv", "config": make_ui_config(
#     kfv_mode, kfv_iterations, kfv_kernel, kfv_delta)}

# # 右侧估计器配置 (Re-FGO)：传入 kfv_mode 供 imitate_kfv 使用
# right = {"name": "Re-FGO", "kind": "fgo", "config": make_ui_config(
#     kfv_mode, fgo_iterations, fgo_kernel, fgo_delta, fgo_imitate_kfv,
#     fgo_window_size, fgo_auto_diff)}

# experiment = run_and_display(sim_data, left, right)
# display(animate_pair(experiment, interval=80))


🚀 [Run Estimator 1]: KFV-EKF (kfv)
🚀 [Run Estimator 2]: Re-FGO (fgo)
  [DEBUG Metrics] 估计轨迹 Shape: (4, 100) | 真值 Shape: (2, 100) | 对齐计算帧数: 100
  [DEBUG Metrics] 估计轨迹 Shape: (4, 100) | 真值 Shape: (2, 100) | 对齐计算帧数: 100

统计结果与维度诊断列表:
KFV-EKF: RMSE=22.493 m | MAE=17.053 m | CP95=27.118 m | Max=141.421 m | Time=1233.76 ms (12.34 ms/step)
Re-FGO: RMSE=19.036 m | MAE=6.894 m | CP95=25.415 m | Max=141.421 m | Time=4533.80 ms (45.34 ms/step)



In [4]:
# @title 3. FGO vs FGO
# @markdown Simulation parameters and both FGO configurations are adjustable. Results play side-by-side below.
from IPython.display import display
import importlib
import experiment_common
importlib.reload(experiment_common)
from experiment_common import animate_pair, make_config, make_data, run_and_display

# --------------------------------------------------
# Simulation Data Parameters
# --------------------------------------------------
steps = 100  # @param {type:"slider", min:30, max:160, step:10}  # Unit: steps
distance = 105  # @param {type:"slider", min:200, max:1000, step:50}  # Unit: m
seed = 7  # @param {type:"integer", min:0, max:999}
outlier_weight = 0.2  # @param {type:"slider", min:0, max:1, step:0.05}  # Ratio (0-1)
outlier_mean = 30.0  # @param {type:"number"}  # Unit: m
outlier_sigma = 5.0  # @param {type:"number"}  # Unit: m
white_sigma = 0.1  # @param {type:"number"}  # Unit: m

# --------------------------------------------------
# Left Column FGO Configuration
# --------------------------------------------------
left_iterations = 2  # @param {type:"integer", min:1, max:20}
left_kernel = "none"  # @param ["none", "huber", "cauchy", "tukey"]
left_delta = 2.0  # @param {type:"number"}
left_imitate_kfv = False  # @param {type:"boolean"}
left_window_size = 1  # @param {type:"integer", min:1, max:20}
left_auto_diff = False  # @param {type:"boolean"}

# --------------------------------------------------
# Right Column FGO Configuration
# --------------------------------------------------
right_iterations = 2  # @param {type:"integer", min:1, max:20}
right_kernel = "huber"  # @param ["none", "huber", "cauchy", "tukey"]
right_delta = 2.0  # @param {type:"number"}
right_imitate_kfv = False  # @param {type:"boolean"}
right_window_size = 20  # @param {type:"integer", min:1, max:20}
right_auto_diff = False  # @param {type:"boolean"}

# --------------------------------------------------
# Helper Functions and Execution
# --------------------------------------------------
def make_ui_config(iterations, kernel, delta, imitate, window, autodiff):
  return make_config(
      err_x=100.0, err_y=-100.0, err_vx=0.0, err_vy=0.0,
      p0_diag=(50.0, 50.0, 1.0, 1.0), kfv_mode="FGO",
      robust_kernel=kernel, robust_delta=delta, max_iteration=iterations,
      window_size=window, imitate_kfv=imitate, autodiff=autodiff
  )

sim_data = make_data(seed=seed, num_steps=steps, anchor_radius=distance,
                     outlier_weight=outlier_weight, outlier_mean=outlier_mean,
                     outlier_sigma=outlier_sigma, white_sigma=white_sigma)

left = {"name": "FGO-config-1", "kind": "fgo", "config": make_ui_config(
    left_iterations, left_kernel, left_delta, left_imitate_kfv, left_window_size, left_auto_diff)}

right = {"name": "FGO-config-2", "kind": "fgo", "config": make_ui_config(
    right_iterations, right_kernel, right_delta, right_imitate_kfv, right_window_size, right_auto_diff)}

experiment = run_and_display(sim_data, left, right)
display(animate_pair(experiment, interval=80))


🚀 [Run Estimator 1]: FGO-config-1 (fgo)
🚀 [Run Estimator 2]: FGO-config-2 (fgo)
  [DEBUG Metrics] 估计轨迹 Shape: (4, 100) | 真值 Shape: (2, 100) | 对齐计算帧数: 100
  [DEBUG Metrics] 估计轨迹 Shape: (4, 100) | 真值 Shape: (2, 100) | 对齐计算帧数: 100

统计结果与维度诊断列表:
FGO-config-1: RMSE=8.931 m | MAE=8.461 m | CP95=13.446 m | Max=18.930 m | Time=17232.67 ms (172.33 ms/step)
FGO-config-2: RMSE=14.142 m | MAE=1.486 m | CP95=0.130 m | Max=141.421 m | Time=6070.15 ms (60.70 ms/step)

